# SpyCloud Simulated Investigation Scenarios
## Purple Team Training & Validation Blueprints

This notebook provides simulated investigation scenarios for training SOC analysts
and validating SpyCloud + Sentinel detection and response capabilities.

**Scenarios:**
1. Infostealer Infection Chain (RedLine Stealer)
2. Executive Email Compromise (credential reuse)
3. Lateral Movement (compromised service account)
4. MFA Bypass (stolen session cookies)
5. Supply Chain Breach (third-party exposure)

Each scenario includes simulated data, detection queries, investigation workflows, and remediation verification.

In [ ]:
# Setup
import pandas as pd
import json
from datetime import datetime, timedelta
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

print('Simulated Scenario Framework loaded.')
print('These scenarios use synthetic data for training purposes.')
print(f'Session started: {datetime.utcnow().isoformat()}Z')

## Scenario 1: Infostealer Infection Chain
**Narrative:** A user clicks a malicious link in a phishing email, downloading a RedLine Stealer payload.
The malware exfiltrates browser credentials, cookies, and autofill data. SpyCloud detects the stolen
credentials on a dark web marketplace 3 days later.

**MITRE ATT&CK:** T1566.002 (Phishing Link) -> T1204.001 (User Execution) -> T1555.003 (Browser Credentials) -> T1539 (Steal Web Session Cookie) -> T1078 (Valid Accounts)

**Expected Detections:**
- SpyCloud Analytics Rule: Infostealer Credentials Detected (Severity 25)
- Defender for Endpoint: Suspicious process execution
- Sentinel Fusion: Multi-stage attack involving credential theft

In [ ]:
# Scenario 1: Simulated infostealer exposure data
scenario1_exposures = pd.DataFrame([
    {'email': 'analyst.training@contoso.com', 'severity': 25, 'password_type': 'plaintext',
     'source_id': 48201, 'breach_title': 'RedLine Stealer Log - March 2026',
     'infected_machine_id': 'SIM-DESKTOP-A1B2C3',
     'infected_path': 'C:\\Users\\jsmith\\AppData\\Local\\Temp\\update.exe',
     'target_url': 'https://login.microsoftonline.com', 'malware_family': 'redline',
     'ip_address': '198.51.100.42'},
    {'email': 'analyst.training@contoso.com', 'severity': 25, 'password_type': 'plaintext',
     'source_id': 48201, 'breach_title': 'RedLine Stealer Log - March 2026',
     'infected_machine_id': 'SIM-DESKTOP-A1B2C3',
     'infected_path': 'C:\\Users\\jsmith\\AppData\\Local\\Temp\\update.exe',
     'target_url': 'https://outlook.office365.com', 'malware_family': 'redline',
     'ip_address': '198.51.100.42'},
    {'email': 'analyst.training@contoso.com', 'severity': 25, 'password_type': 'plaintext',
     'source_id': 48201, 'breach_title': 'RedLine Stealer Log - March 2026',
     'infected_machine_id': 'SIM-DESKTOP-A1B2C3',
     'infected_path': 'C:\\Users\\jsmith\\AppData\\Local\\Temp\\update.exe',
     'target_url': 'https://vpn.contoso.com', 'malware_family': 'redline',
     'ip_address': '198.51.100.42'},
])

display(Markdown('### Scenario 1: Simulated Infostealer Exposure'))
display(Markdown(
    f'- **User:** analyst.training@contoso.com\n'
    f'- **Malware:** RedLine Stealer\n'
    f'- **Severity:** 25 (Critical - stolen sessions + credentials)\n'
    f'- **Infected Device:** SIM-DESKTOP-A1B2C3\n'
    f'- **Targets:** Microsoft 365, Outlook, VPN\n'))
display(scenario1_exposures[['email', 'severity', 'target_url', 'malware_family', 'password_type']])

In [ ]:
# Scenario 1: Detection KQL query
detection_query = """
// SpyCloud Analytics Rule: Infostealer Credentials Detected (Severity 25)
SpyCloudBreachWatchlist_CL
| where severity_d == 25
| where TimeGenerated >= ago(24h)
| where isnotempty(infected_machine_id_s)
| project
    TimeGenerated, email_s, severity_d, password_type_s,
    infected_machine_id_s, infected_path_s, target_url_s,
    malware_family_s = coalesce(malware_family_s, 'Unknown'),
    ip_address_s
| extend AccountEntity = email_s, HostEntity = infected_machine_id_s, IPEntity = ip_address_s
"""

display(Markdown('### Detection KQL Query'))
display(Markdown(f'```kql\n{detection_query}\n```'))

display(Markdown(
    '### Investigation Workflow\n'
    '1. **Triage:** Confirm severity 25 exposure with stolen sessions\n'
    '2. **Scope:** Query SpyCloud for all credentials from same device\n'
    '3. **Correlate:** Check Defender for Endpoint for malware alerts\n'
    '4. **Assess:** Verify sign-in activity since infection date\n'
    '5. **Respond:**\n'
    '   - Force password reset (immediate)\n'
    '   - Revoke all active sessions\n'
    '   - Isolate device in MDE\n'
    '   - Block in Conditional Access\n'
    '   - Scan device for malware remnants\n'
    '6. **Document:** Update Sentinel incident with findings\n'
    '7. **Prevent:** Review phishing training, email filtering rules'))

## Scenario 2: Executive Email Compromise
**Narrative:** A C-level executive's credentials appear in a third-party breach (severity 5).
The same password is reused across multiple services. An attacker uses credential stuffing
to access the executive's email, intercepting sensitive board communications.

**MITRE ATT&CK:** T1589.001 (Gather Credentials) -> T1110.004 (Credential Stuffing) -> T1078.004 (Cloud Accounts) -> T1114.002 (Email Collection)

In [ ]:
# Scenario 2: Simulated executive compromise
scenario2_exposures = pd.DataFrame([
    {'email': 'ciso.training@contoso.com', 'severity': 5,
     'password_type': 'plaintext', 'source_id': 45100,
     'breach_title': 'GenericDB Breach 2025',
     'target_url': 'https://genericservice.example.com',
     'sighting_count': 47},
    {'email': 'ciso.training@contoso.com', 'severity': 2,
     'password_type': 'SHA256', 'source_id': 42300,
     'breach_title': 'Combo List March 2025',
     'target_url': 'https://forum.example.com',
     'sighting_count': 12},
])

display(Markdown(
    '### Scenario 2: Executive Credential Reuse\n'
    '- **User:** ciso.training@contoso.com (C-Suite)\n'
    '- **Risk:** Plaintext password with 47 sightings = high reuse probability\n'
    '- **Detection:** Third-party breach (severity 5) but plaintext available\n\n'
    '**Investigation Steps:**\n'
    '1. Check password against AD (hash comparison via NIST endpoint)\n'
    '2. Review sign-in logs for anomalous locations/IPs\n'
    '3. Check email forwarding rules for unauthorized redirects\n'
    '4. Review OAuth consent grants for suspicious apps\n'
    '5. Verify MFA enrollment status\n'
    '6. Check for data exfiltration indicators'))
display(scenario2_exposures)

## Scenario 3: Lateral Movement via Service Account
**Narrative:** A service account used for application integration appears in an infostealer log.
The account has broad permissions across multiple Azure resources. An attacker uses the stolen
credentials to move laterally across the environment.

**MITRE ATT&CK:** T1078.001 (Default Accounts) -> T1021 (Remote Services) -> T1087 (Account Discovery) -> T1069 (Permission Groups Discovery)

In [ ]:
# Scenario 3: Service account compromise
scenario3_exposures = pd.DataFrame([
    {'email': 'svc-integration@contoso.com', 'severity': 20,
     'password_type': 'plaintext', 'source_id': 49500,
     'breach_title': 'LummaC2 Stealer Log - Feb 2026',
     'infected_machine_id': 'SIM-SERVER-DB01',
     'target_url': 'https://portal.azure.com',
     'malware_family': 'lummac2'},
])

display(Markdown(
    '### Scenario 3: Service Account Lateral Movement\n'
    '- **Account:** svc-integration@contoso.com (Service Account)\n'
    '- **Severity:** 20 (Infostealer credential)\n'
    '- **Risk:** Service accounts often have elevated privileges and no MFA\n\n'
    '**Blast Radius Assessment:**\n'
    '1. List all Azure role assignments for the service account\n'
    '2. Identify all applications/resources using this account\n'
    '3. Check for recent API calls from unexpected IPs\n'
    '4. Review Key Vault access logs\n'
    '5. Check for new service principal credentials created\n\n'
    '**Remediation:**\n'
    '1. Rotate all credentials immediately\n'
    '2. Revoke existing tokens\n'
    '3. Implement Conditional Access for service accounts\n'
    '4. Enable continuous access evaluation (CAE)\n'
    '5. Move to managed identities where possible'))
display(scenario3_exposures)

## Scenario 4: MFA Bypass via Stolen Session Cookies
**Narrative:** An infostealer captures active session cookies from a user's browser.
The attacker imports these cookies into their own browser, bypassing MFA entirely.
SpyCloud detects the stolen cookies via SIP (Session Identity Protection).

**MITRE ATT&CK:** T1539 (Steal Web Session Cookie) -> T1550.004 (Web Session Cookie) -> T1078.004 (Cloud Accounts)

In [ ]:
# Scenario 4: MFA bypass via stolen cookies
scenario4_data = {
    'email': 'finance.user@contoso.com',
    'severity': 25,
    'stolen_cookies': [
        {'domain': '.microsoft.com', 'name': 'ESTSAUTH', 'secure': True},
        {'domain': '.office.com', 'name': 'OIDCAuth', 'secure': True},
        {'domain': '.sharepoint.com', 'name': 'FedAuth', 'secure': True},
    ],
    'infected_machine_id': 'SIM-LAPTOP-FIN01',
    'malware_family': 'vidar',
}

display(Markdown(
    '### Scenario 4: MFA Bypass via Stolen Cookies\n'
    '- **User:** finance.user@contoso.com (Finance Department)\n'
    '- **Severity:** 25 (Critical - session cookies stolen)\n'
    '- **Stolen Cookies:** ESTSAUTH, OIDCAuth, FedAuth\n'
    '- **Impact:** Attacker can access M365, SharePoint without MFA\n\n'
    '**Detection KQL:**\n'
    '```kql\n'
    'SpyCloudSipCookies_CL\n'
    '| where TimeGenerated >= ago(24h)\n'
    '| where cookie_domain_s has ".microsoft.com" or cookie_domain_s has ".office.com"\n'
    '| project TimeGenerated, email_s, cookie_domain_s, cookie_name_s, infected_machine_id_s\n'
    '| extend AccountEntity = email_s\n'
    '```\n\n'
    '**Remediation:**\n'
    '1. Revoke ALL active sessions immediately (revokeSignInSessions)\n'
    '2. Force re-authentication across all apps\n'
    '3. Reset password (attacker may have used session to change it)\n'
    '4. Isolate infected device\n'
    '5. Enable token protection (Conditional Access)\n'
    '6. Implement continuous access evaluation (CAE)'))

## Scenario 5: Supply Chain Breach
**Narrative:** A third-party vendor that manages your SSO integration suffers a breach.
Employee credentials used on the vendor's platform are exposed. Multiple employees
reused their corporate passwords on the vendor site.

**MITRE ATT&CK:** T1199 (Trusted Relationship) -> T1078 (Valid Accounts) -> T1110.004 (Credential Stuffing)

In [ ]:
# Scenario 5: Supply chain breach
scenario5_exposures = pd.DataFrame([
    {'email': 'user1@contoso.com', 'severity': 5, 'password_type': 'plaintext',
     'breach_title': 'VendorCRM Breach 2026', 'sighting_count': 3},
    {'email': 'user2@contoso.com', 'severity': 5, 'password_type': 'plaintext',
     'breach_title': 'VendorCRM Breach 2026', 'sighting_count': 3},
    {'email': 'admin@contoso.com', 'severity': 5, 'password_type': 'MD5',
     'breach_title': 'VendorCRM Breach 2026', 'sighting_count': 3},
    {'email': 'hr.manager@contoso.com', 'severity': 5, 'password_type': 'plaintext',
     'breach_title': 'VendorCRM Breach 2026', 'sighting_count': 3},
    {'email': 'it.support@contoso.com', 'severity': 5, 'password_type': 'SHA1',
     'breach_title': 'VendorCRM Breach 2026', 'sighting_count': 3},
])

display(Markdown(
    '### Scenario 5: Supply Chain Breach\n'
    '- **Source:** VendorCRM Breach 2026\n'
    '- **Affected Users:** 5 employees (could be hundreds in real scenario)\n'
    '- **Risk:** Password reuse across corporate and vendor accounts\n\n'
    '**Investigation Steps:**\n'
    '1. Identify all employees with accounts on breached vendor\n'
    '2. Check password hashes against AD (via NIST password check)\n'
    '3. Identify users with plaintext matches (highest risk)\n'
    '4. Cross-reference with sign-in anomalies post-breach date\n'
    '5. Review vendor access patterns and permissions\n\n'
    '**Remediation:**\n'
    '1. Force password reset for all exposed users\n'
    '2. Disable SSO trust with breached vendor temporarily\n'
    '3. Audit vendor access logs\n'
    '4. Implement password policy preventing corporate password reuse\n'
    '5. Review vendor security posture and SLA requirements'))
display(scenario5_exposures)

## Deployment Validation Checklist
Use this checklist to validate your SpyCloud + Sentinel deployment is detecting and responding correctly.

| # | Check | Expected Result | Status |
|---|-------|----------------|--------|
| 1 | SpyCloud data ingesting to `SpyCloudBreachWatchlist_CL` | Records visible in Log Analytics | |
| 2 | Severity 25 analytics rule fires | Sentinel incident created | |
| 3 | Severity 20 analytics rule fires | Sentinel incident created | |
| 4 | Severity 5 analytics rule fires | Sentinel incident created | |
| 5 | Password reset playbook triggers | User password reset in Entra ID | |
| 6 | Session revocation playbook triggers | Sessions revoked via Graph API | |
| 7 | Device isolation playbook triggers | Device isolated in MDE | |
| 8 | CA blocking playbook triggers | User added to CA block group | |
| 9 | Enrichment playbook adds incident comment | Comment with SpyCloud data visible | |
| 10 | Workbook shows exposure data | All visualizations render correctly | |
| 11 | SCORCH agent responds to queries | Investigation results returned | |
| 12 | AI Engine health check passes | `/api/ai/health` returns healthy | |
| 13 | MCP server connects via SSE | Tools discoverable by AI agents | |
| 14 | Graph Investigation notebook runs | Graph visualization renders | |
| 15 | Executive report generates | PDF-ready output | |